In [2]:
import os
from pathlib import Path

import mysql.connector
import numpy as np
from dotenv import load_dotenv


def trouver_racine_projet():
    dossier_actuel = Path.cwd()

    for dossier in [dossier_actuel, *dossier_actuel.parents]:
        if (dossier / ".env").exists():
            return dossier

    raise FileNotFoundError(
        "Impossible de trouver le fichier .env"
    )


PROJECT_ROOT = trouver_racine_projet()
load_dotenv(PROJECT_ROOT / ".env")

print("Racine du projet :", PROJECT_ROOT)


NOMBRE_LOGEMENTS = 1_000
SEED = 42

QUARTIERS = [
    (1, "Centre", 0.95),
    (2, "Résidentiel", 0.80),
    (3, "Universitaire", 0.65),
    (4, "Périphérie", 0.50),
    (5, "Zone rurale", 0.35),
]


def creer_connexion():
    """Créer une connexion vers MySQL."""

    variables_obligatoires = [
        "DB_HOST",
        "DB_PORT",
        "DB_NAME",
        "DB_USER",
        "DB_PASSWORD",
    ]

    variables_manquantes = [
        variable
        for variable in variables_obligatoires
        if not os.getenv(variable)
    ]

    if variables_manquantes:
        raise RuntimeError(
            f"Variables manquantes dans .env : {variables_manquantes}"
        )

    return mysql.connector.connect(
        host=os.getenv("DB_HOST"),
        port=int(os.getenv("DB_PORT")),
        database=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
    )


def generer_logements(nombre_logements, seed):
    """Générer un dataset immobilier synthétique avec NumPy."""

    rng = np.random.default_rng(seed)

    id_quartier = rng.integers(
        low=1,
        high=len(QUARTIERS) + 1,
        size=nombre_logements,
    )

    surface_m2 = np.round(
        rng.uniform(25, 220, nombre_logements),
        2,
    )

    nombre_pieces = np.rint(
        surface_m2 / 25
        + rng.normal(0, 0.8, nombre_logements)
    ).astype(int)

    nombre_pieces = np.clip(nombre_pieces, 1, 9)

    nombre_chambres = np.rint(
        nombre_pieces * 0.55
        + rng.normal(0, 0.7, nombre_logements)
    ).astype(int)

    nombre_chambres = np.clip(nombre_chambres, 1, 6)
    nombre_chambres = np.minimum(
        nombre_chambres,
        nombre_pieces,
    )

    age_batiment = rng.integers(
        0,
        101,
        nombre_logements,
    )

    distance_centre_km = np.round(
        rng.uniform(0.5, 30, nombre_logements),
        2,
    )

    garage = rng.binomial(
        1,
        0.55,
        nombre_logements,
    )

    possede_jardin = rng.random(nombre_logements) < 0.55

    jardin_m2 = np.where(
        possede_jardin,
        rng.uniform(10, 300, nombre_logements),
        0,
    )

    jardin_m2 = np.round(jardin_m2, 2)

    indices_attractivite = np.array([
        quartier[2]
        for quartier in QUARTIERS
    ])

    attractivite = indices_attractivite[id_quartier - 1]

    bruit = rng.normal(
        loc=0,
        scale=25_000,
        size=nombre_logements,
    )

    prix_eur = (
        30_000
        + 3_200 * surface_m2
        + 12_000 * nombre_pieces
        + 8_000 * nombre_chambres
        - 1_200 * age_batiment
        - 4_500 * distance_centre_km
        + 25_000 * garage
        + 400 * jardin_m2
        + 90_000 * attractivite
        + bruit
    )

    prix_eur = np.maximum(prix_eur, 50_000)
    prix_eur = np.round(prix_eur, 2)

    logements = []

    for index in range(nombre_logements):
        logement = (
            int(id_quartier[index]),
            float(surface_m2[index]),
            int(nombre_pieces[index]),
            int(nombre_chambres[index]),
            int(age_batiment[index]),
            float(distance_centre_km[index]),
            int(garage[index]),
            float(jardin_m2[index]),
            float(prix_eur[index]),
        )

        logements.append(logement)

    return logements


def inserer_donnees(connexion, logements):
    """Insérer les quartiers et les logements dans MySQL."""

    curseur = connexion.cursor()

    requete_quartiers = """
        INSERT IGNORE INTO quartiers
            (id_quartier, nom, indice_attractivite)
        VALUES (%s, %s, %s)
    """

    requete_logements = """
        INSERT INTO logements (
            id_quartier,
            surface_m2,
            nombre_pieces,
            nombre_chambres,
            age_batiment,
            distance_centre_km,
            garage,
            jardin_m2,
            prix_eur
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
    """

    try:
        curseur.execute(
            "SELECT COUNT(*) FROM logements"
        )

        nombre_existant = curseur.fetchone()[0]

        if nombre_existant > 0:
            print(
                f"La table contient déjà "
                f"{nombre_existant} logements."
            )
            print("Aucune nouvelle insertion effectuée.")
            return

        curseur.executemany(
            requete_quartiers,
            QUARTIERS,
        )

        curseur.executemany(
            requete_logements,
            logements,
        )

        connexion.commit()

        print(f"{len(QUARTIERS)} quartiers disponibles.")
        print(f"{len(logements)} logements insérés.")

    except Exception:
        connexion.rollback()
        raise

    finally:
        curseur.close()


def main():
    logements = generer_logements(
        nombre_logements=NOMBRE_LOGEMENTS,
        seed=SEED,
    )

    connexion = creer_connexion()

    try:
        inserer_donnees(
            connexion=connexion,
            logements=logements,
        )
    finally:
        connexion.close()


if __name__ == "__main__":
    main()

Racine du projet : /Users/seresiaka/Downloads/ml_immobilier
5 quartiers disponibles.
1000 logements insérés.
